# Transfer Learning avec PyTorch : CIFAR-10

## Objectif pédagogique

Cette série de notebooks reprend la logique du CNN + Data Augmentation du notebook de référence, puis remplace le CNN construit from scratch par un modèle **pré-entraîné sur ImageNet**.

Nous allons comparer trois stratégies :

1. **Feature Extraction** : backbone gelé, seul le classifieur est entraîné.
2. **Fine-Tuning partiel** : le classifieur et une partie profonde du backbone sont entraînés.
3. **Fine-Tuning complet** : toutes les couches du modèle pré-entraîné sont entraînées.

> Point important : le fine-tuning utilise nécessairement un modèle pré-entraîné. Dire « fine-tuning sans transfer learning » n'est donc pas techniquement cohérent. Dans le troisième notebook, « juste fine-tuning » signifie ici **fine-tuning complet du modèle pré-entraîné**, sans étape séparée de feature extraction.


## 1. Imports et reproductibilité


In [ ]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device utilisé :", device)

## 2. Pourquoi ResNet18 et pourquoi redimensionner CIFAR-10 ?

CIFAR-10 contient des images de **32×32×3**. Le modèle pré-entraîné ResNet18 a été appris sur ImageNet avec des images généralement utilisées en **224×224**.

Nous allons donc :

- conserver l'image RGB ;
- redimensionner vers 224×224 ;
- appliquer une augmentation uniquement au train ;
- normaliser avec les statistiques ImageNet.

Le point pédagogique essentiel est que le backbone pré-entraîné a déjà appris des représentations visuelles : contours, textures, motifs, formes, etc.


In [ ]:
# Dataset utilisé uniquement pour calculer les statistiques
stats_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

stats_loader = DataLoader(
    stats_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0
)

channel_sum = torch.zeros(3)
channel_sum_squared = torch.zeros(3)
num_pixels = 0

for images, _ in stats_loader:
    batch_size, channels, height, width = images.shape
    pixels = batch_size * height * width

    channel_sum += images.sum(dim=(0, 2, 3))
    channel_sum_squared += (images ** 2).sum(dim=(0, 2, 3))
    num_pixels += pixels

IMAGENET_MEAN = (channel_sum / num_pixels).tolist()
IMAGENET_STD = torch.sqrt(
    channel_sum_squared / num_pixels - torch.tensor(IMAGENET_MEAN) ** 2
).tolist()

print("Mean :", IMAGENET_MEAN)
print("Std  :", IMAGENET_STD)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomCrop(224, padding=16),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.15),
        ratio=(0.3, 3.3)
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

## 3. Charger et séparer CIFAR-10

Nous conservons la même logique que le notebook de référence :

- 45 000 images pour le train ;
- 5 000 images pour la validation ;
- 10 000 images officielles pour le test.

Le split est fixé avec `SEED=42` afin que les trois expériences soient comparables.


In [ ]:
full_train_aug = datasets.CIFAR10(
    root="./data", train=True, download=False, transform=train_transform
)

full_train_eval = datasets.CIFAR10(
    root="./data", train=True, download=False, transform=eval_transform
)

test_dataset = datasets.CIFAR10(
    root="./data", train=False, download=False, transform=eval_transform
)

train_size, val_size = 45000, 5000
generator = torch.Generator().manual_seed(SEED)

train_split, val_split = random_split(
    range(len(full_train_aug)),
    [train_size, val_size],
    generator=generator
)

train_indices = train_split.indices
val_indices = val_split.indices

train_dataset = torch.utils.data.Subset(full_train_aug, train_indices)
val_dataset = torch.utils.data.Subset(full_train_eval, val_indices)

print("Train :", len(train_dataset))
print("Validation :", len(val_dataset))
print("Test :", len(test_dataset))

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print("Images :", images.shape)
print("Labels :", labels.shape)


In [ ]:
classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

def unnormalize(img):
    mean_t = torch.tensor(IMAGENET_MEAN, dtype=img.dtype).view(3, 1, 1)
    std_t = torch.tensor(IMAGENET_STD, dtype=img.dtype).view(3, 1, 1)
    return (img.cpu() * std_t + mean_t).clamp(0, 1)

raw_base = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transforms.ToTensor()
)

raw_img, label = raw_base[0]

plt.figure(figsize=(3, 3))
plt.imshow(raw_img.permute(1, 2, 0))
plt.title(f"Image originale : {classes[label]}")
plt.axis("off")
plt.show()

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for ax in axes:
    img, lab = full_train_aug[0]

    img = unnormalize(img)
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(classes[lab])
    ax.axis("off")

plt.suptitle("Versions augmentées de la même image")
plt.tight_layout()
plt.show()

## 4. Charger le modèle pré-entraîné

`weights=ResNet18_Weights.DEFAULT` charge les poids appris sur ImageNet.

Nous remplaçons la dernière couche :

`1000 classes ImageNet → 10 classes CIFAR-10`.

Le backbone est ensuite configuré différemment selon le notebook.


In [ ]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Remplacer le classifieur ImageNet par un classifieur CIFAR-10
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, 10)

model = model.to(device)

print(model)

In [ ]:
dummy = torch.zeros(2, 3, 224, 224, device=device)

with torch.no_grad():
    output = model(dummy)

print("Entrée :", dummy.shape)
print("Sortie :", output.shape)
assert output.shape == (2, 10)

## 5. Loss, évaluation et entraînement

Nous conservons une structure proche du notebook original :

- `CrossEntropyLoss(label_smoothing=0.1)` ;
- `AdamW` ;
- scheduler cosine ;
- sauvegarde du meilleur état selon la validation loss ;
- early stopping.

La différence fondamentale entre les notebooks se trouve dans **les paramètres entraînables** et donc dans l'optimiseur.


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total

# Version 3 : Fine-Tuning complet

## Principe

Nous partons du ResNet18 pré-entraîné sur ImageNet, mais **toutes les couches sont entraînables**.

Il n'y a pas d'étape où le backbone est utilisé comme extracteur fixe.

`Image → ResNet18 pré-entraîné entièrement adaptable → nouveau classifieur → classe`

C'est ce que nous appelons ici **« juste fine-tuning » : fine-tuning complet**.


## 5. Configuration des paramètres entraînables


In [ ]:
# FINE-TUNING COMPLET
# Toutes les couches sont entraînables
for param in model.parameters():
    param.requires_grad = True

trainable_params = [p for p in model.parameters() if p.requires_grad]
total_trainable = sum(p.numel() for p in trainable_params)
total_params = sum(p.numel() for p in model.parameters())

print(f"Paramètres totaux      : {total_params:,}")
print(f"Paramètres entraînables: {total_trainable:,}")
print("Backbone gelé : NON")
print("Fine-tuning complet : OUI")

# LR faible pour éviter de détruire trop rapidement
# les représentations pré-entraînées.
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

EPOCHS = 30
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

## 6. Entraîner le modèle


In [ ]:
history = {
    "train_loss": [], "train_acc": [],
    "val_loss": [], "val_acc": [], "lr": []
}

PATIENCE = 7
MIN_DELTA = 1e-4
best_val_loss = float("inf")
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    if val_loss < best_val_loss - MIN_DELTA:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss={train_loss:.4f} | Train Acc={train_acc:.2%} | "
        f"Val Loss={val_loss:.4f} | Val Acc={val_acc:.2%} | "
        f"LR={current_lr:.7f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping à l'époque {epoch}.")
        break

if best_state is not None:
    model.load_state_dict(best_state)

print(f"Meilleure Val Loss : {best_val_loss:.4f}")

## 7. Courbes d'apprentissage


In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

## 8. Évaluation finale sur le test set


In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Test Loss : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc:.2%}")

In [ ]:
import os

os.makedirs("./model", exist_ok=True)

model_path = "./model/resnet18_cifar10_finetuning_complet.pth"

torch.save({
    "model_state_dict": model.state_dict(),
    "classes": classes,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
    "test_accuracy": test_acc,
}, model_path)

print(f"Modèle sauvegardé dans : {model_path}")

## 9. Comment interpréter les résultats ?

### Cas A : Train et validation progressent ensemble

C'est le comportement attendu : le modèle apprend des représentations utiles et généralise.

### Cas B : Train accuracy très élevée, validation nettement plus faible

Il existe probablement de l'overfitting. Avec le fine-tuning complet, ce phénomène peut apparaître plus facilement car beaucoup plus de paramètres sont modifiés.

### Cas C : Train accuracy reste plus faible mais validation est correcte

Ce n'est pas nécessairement mauvais. L'augmentation des données, le Dropout/regularisation et le label smoothing rendent le problème d'entraînement plus difficile que l'évaluation.

### Cas D : Feature Extraction plus faible que Fine-Tuning

C'est fréquent lorsqu'il existe un **domain shift** entre ImageNet et CIFAR-10. Le backbone ImageNet fournit de bonnes représentations génériques, mais adapter certaines couches au nouveau domaine peut améliorer la performance.

### Comparaison à faire en classe

| Stratégie | Backbone | Paramètres entraînés | Coût | Adaptation |
|---|---|---|---|---|
| Feature Extraction | gelé | faible | faible | faible |
| Fine-Tuning partiel | partiellement dégelé | moyen | moyen | moyenne |
| Fine-Tuning complet | entièrement dégelé | élevé | élevé | forte |

> Pour une comparaison scientifique correcte, gardez le même split, les mêmes transformations, le même nombre maximal d'époques et le même protocole d'évaluation.


## 10. Checklist de comparaison

Après avoir exécuté les trois notebooks, relever :

- meilleur `Val Accuracy` ;
- meilleur `Test Accuracy` ;
- meilleure `Val Loss` ;
- nombre de paramètres entraînables ;
- nombre d'époques avant early stopping ;
- temps d'entraînement.

**Ne comparez pas uniquement l'accuracy.** Une stratégie peut obtenir une accuracy proche avec beaucoup moins de paramètres et beaucoup moins de calcul.
